In [1]:
# ── Mongo candle-coverage audit for the 4 ladder pairs ──────────────────────
import os, datetime as dt
from pymongo import MongoClient

MONGO_IP   = os.environ.get("TRUENAS_LAN_IP", "192.168.1.54")
MONGO_PASS = os.environ.get("MONGO_ROOT_PASSWORD") or "PASTE_PASSWORD_HERE"

PAIRS      = ["XMR-USDT", "DASH-USDT", "SUN-USDT", "ZANO-USDT"]
CONNECTORS = ["nonkyc", "mexc"]          # check both (mexc may proxy nonkyc)
INTERVALS  = ["5m", "1h", "1d"]
_SEC       = {"1m":60,"5m":300,"15m":900,"30m":1800,"1h":3600,"4h":14400,"1d":86400}

cli  = MongoClient(f"mongodb://root:{MONGO_PASS}@{MONGO_IP}:27017/?authSource=admin",
                   serverSelectionTimeoutMS=8000)
db   = cli["quants_lab"]
coll = db["candles"]

# --- 1. sniff one doc to detect field names ---------------------------------
doc = coll.find_one()
if doc is None:
    raise SystemExit("candles collection is empty")
print("SAMPLE DOC KEYS:", sorted(doc.keys()))
def pick(*cands):
    return next((c for c in cands if c in doc), None)
F_PAIR = pick("trading_pair","symbol","pair","market")
F_CONN = pick("connector_name","connector","exchange","source")
F_INT  = pick("interval","timeframe","resolution")
F_TS   = pick("timestamp","ts","time","open_time","t")
print(f"using fields: pair={F_PAIR} conn={F_CONN} interval={F_INT} ts={F_TS}\n")

# normalize a ts sample so we know ms vs s vs datetime
def to_epoch_s(v):
    if isinstance(v, dt.datetime): return v.timestamp()
    v = float(v)
    return v/1000.0 if v > 1e11 else v

# --- 2. what pair-name format does the lake use? ----------------------------
distinct_pairs = coll.distinct(F_PAIR)
hits = [p for p in distinct_pairs
        if any(p.replace("/","-").replace("_","-").upper() == t for t in PAIRS)]
print(f"distinct pairs in lake: {len(distinct_pairs)}  |  ladder-pair matches: {hits}\n")

# --- 3. coverage per (connector, pair, interval) ----------------------------
rows = []
for conn in CONNECTORS:
    for pair in distinct_pairs:
        norm = pair.replace("/","-").replace("_","-").upper()
        if norm not in PAIRS:
            continue
        for iv in INTERVALS:
            q = {F_PAIR: pair, F_INT: iv}
            if F_CONN: q[F_CONN] = conn
            n = coll.count_documents(q)
            if n == 0:
                continue
            first = coll.find(q).sort(F_TS, 1).limit(1)[0][F_TS]
            last  = coll.find(q).sort(F_TS, -1).limit(1)[0][F_TS]
            f_s, l_s = to_epoch_s(first), to_epoch_s(last)
            days     = (l_s - f_s) / 86400
            expected = int((l_s - f_s) / _SEC[iv]) + 1
            gap_pct  = (1 - n / expected) * 100 if expected else 0
            stale_d  = (dt.datetime.now(dt.timezone.utc).timestamp() - l_s) / 86400
            rows.append((conn, norm, iv, n, round(days,1), expected,
                         round(gap_pct,2), round(stale_d,1)))

print(f"{'conn':<8}{'pair':<11}{'iv':<5}{'bars':>9}{'days':>8}{'expected':>10}"
      f"{'gap%':>7}{'stale_d':>9}")
for r in rows:
    print(f"{r[0]:<8}{r[1]:<11}{r[2]:<5}{r[3]:>9}{r[4]:>8}{r[5]:>10}{r[6]:>7}{r[7]:>9}")
if not rows:
    print("NO ROWS — pair naming or connector field didn't match; "
          "send me the SAMPLE DOC KEYS output and a full distinct-pairs list.")
print(f"done")

OperationFailure: Authentication failed., full error: {'ok': 0.0, 'errmsg': 'Authentication failed.', 'code': 18, 'codeName': 'AuthenticationFailed'}